<a href="https://colab.research.google.com/github/azholl/lis5693/blob/main/lab-6/lab-6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing and importing libraries

In [ ]:
!pip install textnets

In [ ]:
import textnets as tn

Fixing the initial seed for the pseudorandom number generator

In [ ]:
tn.params

Helps ensure reproducibility

In [ ]:
tn.params["seed"] = 42

Preparing corpus by importing data and saving as df and using tn.Corpus(df["Abstract"]) to pull in just the abstract, since that's the column I focused on in the last lab.

In [ ]:
import requests
import io

url = "https://raw.githubusercontent.com/azholl/lis5693/main/lab-6/lab-6-data.csv"
response = requests.get(url)
response.raise_for_status()
text = response.text

import pandas as pd

df = pd.read_csv(io.StringIO(text))
df.head()

In [ ]:
corpus = tn.Corpus(df["Abstract"], lang="en")

In [ ]:
corpus

Create network, had to add mini_docs=3 (instead of 1) to drop terms that only appear in 1 document, and remove_weak_edges=True because my clusters were unreadable with how many terms and edges there were.

In [ ]:
t = tn.Textnet(corpus.tokenized(), min_docs=3, remove_weak_edges=True)
t

There are 144 terms and 622 edges in my dataset after cleaning it up

Visualize and analyze

In [ ]:
t.plot(label_nodes=True,
       show_clusters=True)

Visualize again but scale the nodes according to their birank and the edges according to their weights

In [ ]:
t.plot(label_nodes=True,
       show_clusters=True,
       scale_nodes_by="birank",
       scale_edges_by="weight")

Using the one-node term-to-term projection, clusters can be interpreted as indicating latent themes

In [ ]:
terms = t.project(node_type=tn.TERM, connected=True)
terms.top_cluster_nodes()

Visualize projected networks

In [ ]:
papers = t.project(node_type=tn.DOC)
papers.plot(label_nodes=True)

Visualizing the term network

In [ ]:
words = t.project(node_type=tn.TERM)
words.plot(label_nodes=True,
           show_clusters=True)

In [ ]:
papers.top_betweenness()

In [ ]:
words.top_betweenness()

In [ ]:
words.plot(label_nodes=True,
           scale_nodes_by="betweenness",
           color_clusters=True,
           alpha=0.5,
           edge_width=[10*w for w in words.edges["weight"]],
           edge_opacity=0.4,
           node_label_filter=lambda n: n.betweenness() > words.betweenness.median())